# GPT-2 small을 NumPy로 직접 돌려보기

이 노트북은 [블로그 실습](https://backedegg.github.io/posts/numpy-gpt2/)에서 사용한 코드를 코랩에서 순서대로 실행한다. **런타임 → 모두 실행**을 누르면 공개 GPT-2 small 가중치 약 550MB를 내려받는다. 처음 실행은 다운로드와 모델 로딩 때문에 시간이 걸린다. CPU 런타임에서도 실행할 수 있다.

코드의 순전파는 NumPy로 계산한다. Hugging Face와 PyTorch는 **기준 출력과 비교하는 셀**에서 사용한다. 각 셀의 결과 바로 아래에 확인할 개념을 적었다.

## 0. 실습 준비

먼저 필요한 패키지를 설치한다. 코랩에 이미 설치된 PyTorch는 그대로 쓰고, 코드와 함께 검증한 라이브러리 버전을 설치한다.

In [ ]:
%pip -q install "numpy==2.1.3" "transformers==4.45.2" "tiktoken==0.8.0" "safetensors==0.4.5" "huggingface-hub==0.36.2" "matplotlib==3.10.7" "regex

실습 코드는 블로그에 첨부된 ZIP을 내려받는다. SHA-256을 확인하므로 내려받은 파일이 이번 글에 사용한 코드와 같은지 알 수 있다.

In [ ]:
import hashlib, io, os, sys, urllib.request, zipfile
from pathlib import Path

LAB = Path('/content/gpt2_numpy_lab')
LAB.mkdir(exist_ok=True)
URL = 'https://backedegg.github.io/posts/numpy-gpt2/gpt2-lab.zip'
EXPECTED_SHA256 = 'afa5520acd8714467a7e1e5c49803f56eede818a7e82dafce09fe0f25f6fe91f'
bundle = urllib.request.urlopen(URL, timeout=60).read()
actual = hashlib.sha256(bundle).hexdigest()
assert actual == EXPECTED_SHA256, f'실습 파일이 달라졌습니다: {actual}'
allowed = {'gpt2_numpy.py', 'download_model.py', 'parity_probe.py',
           'experiments.py', 'bpe_and_losses.py'}
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for name in allowed:
        (LAB / name).write_bytes(archive.read(name))
os.chdir(LAB)
sys.path.insert(0, str(LAB))
print('실습 코드 확인 완료:', ', '.join(sorted(allowed)))

공개 모델을 내려받는다. 실습 코드가 **가중치의 리비전과 SHA-256**도 검사한다.

In [ ]:
from download_model import main as download_model
download_model()

## 1. 문장 → 토큰 ID

`The cat sat on the`를 넣고 실제 토큰 조각과 ID를 함께 보자. 두 번째 토큰의 앞 공백도 토큰의 일부다.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
prompt = 'The cat sat on the'
ids = enc.encode(prompt)
print('토큰 ID:', ids)
print('토큰 조각:', [enc.decode([i]) for i in ids])
assert ids == [464, 3797, 3332, 319, 262]

같은 GPT-2 토크나이저로 한국어와 영어 예문 두 쌍의 토큰 수를 센다. 여기서 비교하는 것은 **이 문장들의 이 토크나이저 결과**다.

In [ ]:
pairs = [('나는 오늘 학교에 간다.', 'I go to school today.'),
         ('고양이가 창문 옆에 앉아 있다.', 'A cat is sitting by the window.')]
for ko, en in pairs:
    print(f'{ko} → {len(enc.encode(ko))}개 | {en} → {len(enc.encode(en))}개')

## 2. BPE 규칙을 직접 적용하기

먼저 `banana`와 `bandana`로 글자 쌍을 합치는 연습을 한다. 그다음 공개 GPT-2 병합 규칙을 읽는 `ByteBPE` 구현과 `tiktoken`의 ID가 같은지 확인한다.

In [ ]:
from experiments import toy_bpe
for step, result in enumerate(toy_bpe(['banana', 'bandana', 'banana']), 1):
    print(step, result['pair'], '등장', result['count'], '번 →', result['pieces'][0])

from bpe_and_losses import ByteBPE
custom = ByteBPE(LAB / 'model')
for text in [pairs[0][0], pairs[0][1], 'The cat sat on the floor.']:
    assert custom.encode(text) == enc.encode(text)
print('직접 구현한 GPT-2 byte BPE와 tiktoken의 ID가 일치합니다.')

## 3. 토큰 임베딩 + 위치 임베딩

ID는 표에서 행을 찾는 번호다. 토큰 표에서 다섯 행, 위치 표에서 0~4번 행을 꺼내 더하면 `(5, 768)` 모양이 된다.

In [ ]:
import numpy as np
from gpt2_numpy import GPT2NumPy
model = GPT2NumPy(LAB / 'model')
token_vectors = model.w['wte.weight'][ids]
position_vectors = model.w['wpe.weight'][np.arange(len(ids))]
x = token_vectors + position_vectors
print('토큰 벡터:', token_vectors.shape)
print('위치 벡터:', position_vectors.shape)
print('더한 결과:', x.shape)
assert x.shape == (5, 768)

## 4. Self-Attention과 causal mask

아래 그림은 실제 모델의 **첫 번째 블록, 첫 번째 머리**의 가중치다. 세로축은 읽는 위치, 가로축은 참고하는 위치다. 대각선 위쪽은 뒤쪽 토큰이라서 0이 된다.

In [ ]:
import matplotlib.pyplot as plt
attn_ids = enc.encode('The cat sat on the mat.')
_, attention = model.forward(attn_ids, attention_layer=0, attention_head=0)
labels = [enc.decode([i]) for i in attn_ids]
plt.figure(figsize=(7, 6))
plt.imshow(attention, vmin=0, vmax=1, cmap='viridis')
plt.xticks(range(len(labels)), labels, rotation=45)
plt.yticks(range(len(labels)), labels)
plt.xlabel('참고하는 토큰')
plt.ylabel('읽는 위치')
plt.colorbar(label='attention weight')
plt.show()
print('어텐션 맵:', attention.shape)
print('뒤쪽 위치 가중치 합:', np.triu(attention, 1).sum())
assert np.triu(attention, 1).sum() == 0

## 5. 블록 12개 → 전체 어휘 점수

`model.forward()` 안에서는 LayerNorm, 12개 머리의 어텐션, 잔차 덧셈, FFN을 **12번** 거친다. 마지막 위치의 50,257개 점수에서 최고 점수 토큰을 꺼낸다.

In [ ]:
logits, _ = model.forward(ids)
next_id = int(logits[-1].argmax())
print('전체 점수 모양:', logits.shape)
print('최고 점수 다음 토큰:', next_id, repr(enc.decode([next_id])))
assert logits.shape == (5, 50257)

## 6. Hugging Face와 전체 logits 비교

생성된 단어 하나만 비교하면 계산 오류를 놓칠 수 있다. 영어와 한국어 입력에서 **모든 위치 × 모든 어휘** 점수의 최대 절대 오차를 계산한다.

In [ ]:
from parity_probe import main as compare_with_huggingface
compare_with_huggingface()

## 7. 샘플링 비교

같은 마지막 위치 점수에서 온도, top-k, top-p만 바꾼다. `floor`가 1등이어도 온도 1.0에서는 그 확률이 약 7.64%다.

In [ ]:
from gpt2_numpy import sampling_distribution
last = logits[-1]
for name, options in [('greedy', None), ('T=0.1', {'temperature': 0.1}),
                      ('T=1.0', {'temperature': 1.0}), ('T=2.0', {'temperature': 2.0}),
                      ('top-k=20', {'top_k': 20}), ('top-p=0.9', {'top_p': 0.9})]:
    if options is None:
        print(name, repr(enc.decode([int(last.argmax())])))
    else:
        p = sampling_distribution(last, **options)
        print(name, '후보', np.count_nonzero(p), 'floor 확률', round(100 * p[4314], 3), '%')

## 8. 부품을 하나씩 뺀 계산

기존 가중치를 그대로 두고 √64 스케일링, causal mask, 잔차 덧셈을 **각각 한 번씩** 제거한다. 마지막 위치 logits가 얼마나 달라지는지 확인한다.

In [ ]:
for name, options in [('스케일링 제거', {'no_scale': True}),
                      ('마스크 제거', {'no_mask': True}),
                      ('잔차 덧셈 제거', {'no_residual': True})]:
    changed, _ = model.forward(ids, **options)
    chosen = int(changed[-1].argmax())
    print(name, '최대 점수 변화', round(float(np.abs(changed[-1] - last).max()), 4),
          '다음 토큰', repr(enc.decode([chosen])))

## 9. 손실 계산과 SFT·RLHF·DPO

`The cat sat on the floor.`에서 실제 다음 토큰에 준 확률을 읽어 교차 엔트로피를 계산한다. 문장 전체의 평균과 답변 ` floor.`에 해당하는 두 칸의 평균을 구분한다. 아래의 보상 모델·PPO·DPO 숫자는 **원리를 보기 위해 정한 작은 예시**다.

In [ ]:
from bpe_and_losses import main as calculate_losses
calculate_losses()

## 직접 문장을 입력해 보기

한 번에 다음 토큰 하나를 계산해 본다. GPT-2 small은 영어 텍스트를 이어 쓰는 데 적합한 기본 모델이다. 한국어 입력도 가능하지만 결과 품질은 다를 수 있다.

In [ ]:
my_prompt = 'The cat sat on the'  # 원하는 문장으로 바꾸기
my_ids = enc.encode(my_prompt)
my_logits, _ = model.forward(my_ids)
candidate_ids = np.argsort(my_logits[-1])[-5:][::-1]
candidate_probs = sampling_distribution(my_logits[-1])
for token_id in candidate_ids:
    print(repr(enc.decode([int(token_id)])), f'{100*candidate_probs[token_id]:.3f}%')

실험 기록은 왼쪽 파일 목록의 `gpt2_numpy_lab` 폴더에 저장된다. `parity_results.json`, `experiment_results.json`, `loss_results.json`에서 각 단계의 원자료를 확인할 수 있다. 구조 설명은 [GPT-2 논문](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)과 [Hugging Face GPT-2 문서](https://huggingface.co/docs/transformers/model_doc/gpt2)를 참고했다.